# A3d — is a granule a random sample of the RNA around it?

> **Reviewer #2, major point 9.** *"The authors should consider whether the detected granules could
> instead reflect locally elevated ambient RNA rather than genuine subcellular compartments."*

Section 5 of `A3c_de_baseline.ipynb` answers this with **summary statistics**: E(g), a share-ratio
between the granule and residual compartments fitted over 50 µm squares, and R(g), a whole-section
regression residual. Both show that granules hold proportionally more neuronal RNA and less glial
RNA than the material beside them. Neither, however, states the reviewer's hypothesis as a
**model of chance** and rejects it.

This notebook does exactly that. The hypothesis has a literal form:

> a granule's contents are a random sample of the non-somatic RNA in its immediate neighbourhood.

That is a sampling statement, so it has a null distribution. Pool each granule with the RNA sitting
right beside it, randomly relabel which of those transcripts are "in the granule" — keeping the
number the granule actually holds — and ask whether the observed neuronal enrichment and glial
depletion are inside the range that produces.

**Two things make this stronger than section 5.**

1. **The neighbourhood is 10 µm, not 50 µm.** Twenty-five times less area, so the standing
   objection — *granules are found in neuron-rich neuropil, and neuropil is full of neuronal RNA* —
   has correspondingly less room to operate. The grid is built from scratch here; there is no
   published 10 µm spot object.
2. **The answer is a probability, not a ranking.** Section 5 reports that neuronal genes rank above
   glial ones. This reports whether the observed separation is one the random-sampling hypothesis
   can produce at all.

Everything is measured on the **252 neutral genes** — the panel genes that neither seeded the
detection nor were used to filter it — so no part of the result follows from how a granule was
defined. See `a3_common.neutral_genes`.

### One null, and why this one

An earlier draft carried a second, *literal* arm: treat the residual composition as known and draw
`Multinomial(N_b, p_b)`. It was retired. It assumes away the uncertainty in a composition estimated
from a finite local pool, so its p-values are anticonservative — on exchangeable data its z-scores
came out about 1.4× too wide where the permutation's were unit-spread. Nothing reported here ever
came from it. Its other purpose, stating the hypothesis as something you could physically
*generate*, is now served far better by **A3e**, which builds ambient pseudo-granules and runs the
detector over them.

### Where this sits in the run

After `A3c_de_baseline.ipynb`, whose section 1 writes the per-transcript compartment labels this
notebook reuses. It needs nothing else: not the Set 0/1/3 detections, not the vicinity controls.

Run from `R2_revision/ambient_controls/`, on the `mcDETECT-env` kernel.

## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
import zlib
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import norm

sys.path.insert(0, str(Path.cwd()))          # run this notebook from ambient_controls/
import a3_config as C
import a3_common as A3

warnings.filterwarnings("ignore")

# -------------------- runtime gates -------------------- #
# THE DEFAULTS BELOW PRODUCE THE FINAL TABLES. Run top to bottom, once, and change nothing.
OVERWRITE = False        # True -> rebuild the 10 um bin counts from the transcript tables
VALIDATE = True          # section 6 correctness gates. On by default: they run last, so every
                         #   table is already written, and gates (c) and (e) are what make the
                         #   closed form believable.

GRID = C.LOCAL_NULL_GRID
MODE = C.LOCAL_NULL_MODES[0]            # "permutation" -- the only null
MIN_POOL = C.LOCAL_NULL_MIN_POOL
T_DRAWS = C.LOCAL_NULL_T_DRAWS

C.ensure_dirs()
OUT = C.A3D_DIR
GENES = A3.neutral_genes()

assert C.LOCAL_NULL_MODES == ["permutation"], C.LOCAL_NULL_MODES

print("writing to      :", OUT)
print(f"neighbourhood   : {GRID:g} um square, pooling all {len(C.Z_GRID)} z-planes "
      f"({C.SPOT_GRID} um in A3c section 5)")
print(f"genes           : {len(GENES)} neutral (of {len(A3.load_genes())} panel)")
print(f"null            : {MODE}   min local pool: {MIN_POOL}")
print(f"T null          : {T_DRAWS:,} parametric draws from the per-gene moments")
print("\nGROUP STATISTIC:", C.LOCAL_NULL_GROUP_STAT)

writing to      : /Users/chenyang/Desktop/mcDETECT/R2_revision/ambient_controls/output/a3d
neighbourhood   : 10 um square, pooling all 7 z-planes (50 um in A3c section 5)
genes           : 252 neutral (of 290 panel)
null            : permutation   min local pool: 20
T null          : 100,000 parametric draws from the per-gene moments

GROUP STATISTIC: median log2(obs/exp) in neuronal genes minus the same in glial and vascular genes, both from the panel's own annotation sheet


## 1. The 10 µm grid, built from scratch

Every transcript already carries a compartment label from A3c — `intrasomatic` if it overlaps a
nucleus, `granule` if it lies inside a published granule sphere, `residual_extrasomatic` otherwise.
That labelling is the single most expensive operation in A3 (10⁸ transcripts against 10⁶ spheres)
and it is **cached and reused here, not recomputed**. The cache is one `int8` column in the
transcript table's own row order, so pairing it back is a positional concatenation; gate (a) in
section 6 checks the totals against `partition_counts.csv` rather than trusting that silently.

The grid itself is new. Bins are 10 × 10 µm squares floored from the section's own minimum
coordinate:

```
ix = floor((x - x_min) / 10),  iy = floor((y - y_min) / 10),  bin = ix * (iy_max + 1) + iy
```

Two deliberate choices:

- **Floored from the section origin, not rounded onto published spot centres.** A3c's 50 µm grid
  uses `np.round((x - sx.min()) / 50)` so that it lands on the spots object whose subtraction it
  was quantifying. There is no published 10 µm object, so there is nothing to align to.
- **Two-dimensional, pooling all seven z-planes**, as the 50 µm grid also does. The section is 9 µm
  deep, so a 10 × 10 µm column through it is close to isotropic; splitting z would buy no extra
  locality and would thin the local pool sevenfold.

Only the 252 neutral genes and the two non-somatic compartments enter. The intrasomatic compartment
plays no part in this analysis at all — the question is entirely about how granule RNA compares
with the RNA immediately outside it.

In [2]:
bin_path = OUT / "a3d_bin_layer_counts.parquet"
if bin_path.exists() and not OVERWRITE:
    binned = pd.read_parquet(bin_path)
    print(f"read cached bin counts: {len(binned):,} rows")
else:
    binned = pd.concat([A3.bin_transcripts(s, grid=GRID, genes=GENES) for s in C.SAMPLES],
                       ignore_index=True)
    A3.write_parquet_atomic(binned, bin_path)
    print(f"wrote {bin_path.name}: {len(binned):,} rows")

print()
print(binned.groupby(["sample", "layer"], observed=True)
      .agg(n_cells=("n", "size"), n_transcripts=("n", "sum"), n_bins=("bin", "nunique")))

read cached bin counts: 29,464,842 rows



                               n_cells  n_transcripts  n_bins
sample layer                                                 
WT     granule                 1675112        2369397  145760
       residual_extrasomatic  15276816       44924390  248343
AD     granule                 1043048        1411344  114808
       residual_extrasomatic  11469866       30321566  224106


## 2. Which bins can carry a null, and what they cover

A bin contributes to the test only if it has something to test and something to test it against:

- at least one granule transcript, `N_b > 0` — otherwise there is nothing to redraw;
- at least `LOCAL_NULL_MIN_POOL` residual transcripts, `n_b ≥ 20` — otherwise the surrounding
  material is too thin for "the RNA around this granule" to mean anything.

The permutation null does not strictly need the second rule: it carries the uncertainty in the
local composition rather than assuming it away, so it stays calibrated however small the pool is.
The rule is kept anyway, because a bin with three residual transcripts supports no *claim* about a
neighbourhood even when the arithmetic is valid, and because it is the rule the scope table below
was written against.

**The coverage this leaves is reported, not buried.** If the surviving bins held only a small
fraction of the granule layer, the result would be a statement about a subset of granules and would
have to be described as one. `a3d_local_null_scope.csv` records it.

In [3]:
mats, scope = {}, []
for s in C.SAMPLES:
    bins, O_all, R_all = A3.local_null_matrices(binned[binned["sample"] == s], GENES)
    N_all = np.asarray(O_all.sum(axis=1)).ravel()
    n_all = np.asarray(R_all.sum(axis=1)).ravel()

    keep = (N_all > 0) & (n_all >= MIN_POOL)
    O, R = O_all[keep], R_all[keep]
    mats[s] = dict(bins=bins[keep], O=O, R=R,
                   N=np.asarray(O.sum(axis=1)).ravel(),
                   n=np.asarray(R.sum(axis=1)).ravel())

    pool = n_all[N_all > 0]
    scope.append(dict(
        sample=s, grid_um=GRID, min_pool=MIN_POOL, n_genes=len(GENES),
        mode=MODE, n_draw_T=T_DRAWS, seed=C.LOCAL_NULL_SEED,
        n_bin_occupied=len(bins), n_bin_with_granule=int((N_all > 0).sum()),
        n_bin_kept=int(keep.sum()),
        granule_tx_total=int(N_all.sum()), granule_tx_kept=int(N_all[keep].sum()),
        frac_granule_tx_kept=float(N_all[keep].sum() / N_all.sum()),
        residual_tx_kept=int(n_all[keep].sum()),
        pool_q05=float(np.quantile(pool, 0.05)), pool_median=float(np.median(pool)),
        pool_q95=float(np.quantile(pool, 0.95)),
        granule_per_bin_median=float(np.median(N_all[keep])),
    ))

scope = pd.DataFrame(scope)
scope.to_csv(OUT / "a3d_local_null_scope.csv", index=False)
print(scope.T.to_string())
for r in scope.itertuples():
    print(f"\n[{r.sample}] kept {r.n_bin_kept:,} of {r.n_bin_with_granule:,} granule-bearing bins, "
          f"covering {r.frac_granule_tx_kept:.2%} of granule transcripts; "
          f"local pool median {r.pool_median:,.0f}")

                                  0            1
sample                           WT           AD
grid_um                        10.0         10.0
min_pool                         20           20
n_genes                         252          252
mode                    permutation  permutation
n_draw_T                     100000       100000
seed                              0            0
n_bin_occupied               248352       224106
n_bin_with_granule           145760       114808
n_bin_kept                   145071       114357
granule_tx_total            2369397      1411344
granule_tx_kept             2367460      1410010
frac_granule_tx_kept       0.999182     0.999055
residual_tx_kept           33354975     19019291
pool_q05                       81.0        59.35
pool_median                   216.0        159.0
pool_q95                      418.0        302.0
granule_per_bin_median          8.0          6.0

[WT] kept 145,071 of 145,760 granule-bearing bins, covering 99.92% o

## 3. The null

Pool bin *b*'s granule and residual extrasomatic transcripts, then randomly relabel which $N_b$ of
them are "granule". With $K_{bg} = O_{bg} + c_{bg}$ the pooled count of gene *g* and
$M_b = N_b + n_b$ the pooled total:

$$X_b \sim \mathrm{MultivariateHypergeometric}(K_b,\ N_b)$$

This is the reviewer's hypothesis stated as a model of chance, and it is the **conservative** way to
state it: the granule's own transcripts sit in the pool it is measured against, so under the
alternative the null dilutes the very contrast it is testing. Whatever it reports is therefore an
understatement.

Bins are independent, so the totals over bins have a closed form, and that is what the per-gene
p-values are computed from:

$$E_g=\sum_b N_b \frac{K_{bg}}{M_b},\qquad
V_g=\sum_b N_b \frac{K_{bg}}{M_b}\Bigl(1-\frac{K_{bg}}{M_b}\Bigr)\frac{M_b-N_b}{M_b-1}$$

The last factor is the finite-population correction. Without it the p-values are anticonservative in
exactly the dense bins that carry most of the weight, so it is not a refinement.

A closed form is only worth as much as its derivation, and two implementations of one formula
agreeing would prove nothing about whether the formula is right. Gate **(c)** therefore does the
dumb thing on real data — physically shuffles labels in a few thousand real bins and checks that
the simulated mean and spread land on $E$ and $\sqrt{V}$.

In [4]:
moments = {}
for s in C.SAMPLES:
    m = mats[s]
    moments[s] = A3.local_null_moments(m["O"], m["R"], MODE)
    E, V = moments[s]
    print(f"[{s}] closed-form moments over {m['O'].shape[0]:,} bins x {len(GENES)} genes; "
          f"expected granule total {E.sum():,.0f} (observed {int(m['N'].sum()):,})")

[WT] closed-form moments over 145,071 bins x 252 genes; expected granule total 2,367,460 (observed 2,367,460)


[AD] closed-form moments over 114,357 bins x 252 genes; expected granule total 1,410,010 (observed 1,410,010)


## 4. Per-gene result

For each gene, `obs` is the number of its transcripts actually found in granules (in the kept
bins), and `exp` is how many the null puts there. The effect size is

$$\log_2\frac{\mathrm{obs}+0.5}{\mathrm{exp}+0.5}$$

and the test statistic is $z=(\mathrm{obs}-\mathrm{exp})/\sqrt{V_g}$, two-sided, with
Benjamini–Hochberg control across the 252 genes within each section and null.

**Two counts are reported, and the second is the one that matters.** The granule layer holds
millions of transcripts, so a compositional shift of a fraction of a percent clears any p-value
threshold; a table saying "251 of 252 genes are significant" would be true and nearly
uninformative. Beside it, therefore, is the number of genes whose observed/expected ratio exceeds
`LOCAL_NULL_EFFECT_THR` in either direction — genes that are not merely detectably but
*materially* over- or under-represented relative to their own surroundings.

In [5]:
panel = A3.load_panel().rename(columns={"Gene": "gene"})
annot = panel[["gene"] + list(C.PANEL_ANNOT_COLS.values())].rename(
    columns={v: k for k, v in C.PANEL_ANNOT_COLS.items()})

rows = []
for s in C.SAMPLES:
    obs = np.asarray(mats[s]["O"].sum(axis=0)).ravel()
    E, V = moments[s]
    sd = np.sqrt(V)
    z = np.divide(obs - E, sd, out=np.zeros_like(E), where=sd > 0)
    d = pd.DataFrame(dict(
        sample=s, mode=MODE, gene=GENES, obs=obs.astype(np.int64), exp=E,
        log2_obs_over_exp=np.log2((obs + 0.5) / (E + 0.5)),
        null_sd=sd, z=z, pval=2.0 * norm.sf(np.abs(z))))
    d["fdr"] = A3.bh_fdr(d["pval"])
    rows.append(d)

genes_tab = pd.merge(pd.concat(rows, ignore_index=True), annot, on="gene", how="left")
genes_tab.to_csv(OUT / "a3d_local_null_genes.csv", index=False)

thr = np.log2(C.LOCAL_NULL_EFFECT_THR)
summ = (genes_tab.assign(sig=lambda d: d["fdr"] < 0.05,
                         big=lambda d: d["log2_obs_over_exp"].abs() > thr,
                         up=lambda d: d["log2_obs_over_exp"] > 0)
        .groupby(["sample", "mode"])
        .agg(n_genes=("gene", "size"), n_fdr05=("sig", "sum"),
             n_above_thr=("big", "sum"), n_enriched=("up", "sum"),
             median_log2=("log2_obs_over_exp", "median"))
        .reset_index())
print(f"genes materially shifted = |log2 obs/exp| > log2({C.LOCAL_NULL_EFFECT_THR}) = {thr:.3f}\n")
print(summ.to_string(index=False))

genes materially shifted = |log2 obs/exp| > log2(1.25) = 0.322

sample        mode  n_genes  n_fdr05  n_above_thr  n_enriched  median_log2
    AD permutation      252      166           46         122    -0.008394
    WT permutation      252      208           51         144     0.047796


## 5. The statistic that answers the reviewer

If granules were random samples of their surroundings, they would hold neuronal and glial RNA in
whatever proportions the surroundings offer. The single number that captures whether they do is the
gap between the two groups:

$$T=\operatorname{median}\bigl(\log_2 \tfrac{\text{obs}}{\text{exp}}\bigr)_{\text{neuronal}}
-\operatorname{median}\bigl(\log_2 \tfrac{\text{obs}}{\text{exp}}\bigr)_{\text{glial + vascular}}$$

The labels come from the probe panel's own design spreadsheet, written when the panel was chosen
and before any granule existed. Nothing in mcDETECT contributed to them, and the detector has no
access to which genes are neuronal.

$T$ is a median of medians and has no closed form, so it is referred to a **parametric null** built
from the per-gene moments of section 3: independent normals with mean $E_g$ and variance $V_g$,
drawn `LOCAL_NULL_T_DRAWS` times, pushed through the same $\log_2(\text{obs}/\text{exp})$ and the
same two group medians. Gene-level independence is the only extra assumption, and it is a mild one
here — the multinomial coupling between 252 genes within a bin is $O(1/252)$ per pair, and $T$ is a
difference of medians over 17 and 39 genes, which is insensitive to it.

The question is whether the observed value lies inside that distribution.

In [6]:
def group_masks(sample):
    g = genes_tab[genes_tab["sample"] == sample]
    g = g.set_index("gene").loc[GENES]                       # pin to the matrix column order
    return (g["cell_type"].isin(C.NONSEED_NEURONAL).to_numpy(),
            g["cell_type"].isin(C.NONSEED_GLIAL).to_numpy())


def contrast(lfc, a, b):
    return float(np.median(lfc[a]) - np.median(lfc[b]))


grows, tnull_keep = [], {}
for s in C.SAMPLES:
    a, b = group_masks(s)
    obs = np.asarray(mats[s]["O"].sum(axis=0)).ravel()
    # crc32, never hash(): Python's string hash is salted per process, so hash() would give a
    # different null on every kernel restart. Same convention as A3a's re-placement null.
    rng = np.random.default_rng(zlib.crc32(f"{C.LOCAL_NULL_SEED}|parametric|{s}".encode()))

    E, V = moments[s]
    lfc = np.log2((obs + 0.5) / (E + 0.5))
    T_obs = contrast(lfc, a, b)

    # Only the annotated genes are drawn -- T depends on nothing else, and drawing all 252 columns
    # costs 200 MB per call for values that are immediately discarded.
    sel = a | b
    Es, Vs, a_s, b_s = E[sel], V[sel], a[sel], b[sel]
    draw = rng.normal(Es, np.sqrt(Vs), size=(T_DRAWS, int(sel.sum())))
    Lp = np.log2((np.maximum(draw, 0.0) + 0.5) / (Es + 0.5))
    Tp = np.median(Lp[:, a_s], axis=1) - np.median(Lp[:, b_s], axis=1)
    del draw, Lp
    tnull_keep[s] = Tp

    grows.append(dict(
        sample=s, mode=MODE, n_neuronal=int(a.sum()), n_glial=int(b.sum()),
        median_neuronal=float(np.median(lfc[a])), median_glial=float(np.median(lfc[b])),
        n_neuronal_above0=int((lfc[a] > 0).sum()), n_glial_above0=int((lfc[b] > 0).sum()),
        T_obs=T_obs, n_draw=int(T_DRAWS),
        T_null_mean=float(Tp.mean()), T_null_sd=float(Tp.std(ddof=1)),
        T_null_max=float(np.abs(Tp).max()),
        z=float((T_obs - Tp.mean()) / Tp.std(ddof=1)),
        p=float(max((np.abs(Tp - Tp.mean()) >= abs(T_obs - Tp.mean())).mean(),
                    1.0 / (len(Tp) + 1)))))

group = pd.DataFrame(grows)
group.to_csv(OUT / "a3d_local_null_group.csv", index=False)
print(group.T.to_string())

print()
for r in group.itertuples():
    print(f"[{r.sample}] T = {r.T_obs:+.3f}  "
          f"(neuronal {r.median_neuronal:+.3f}, {r.n_neuronal_above0}/{r.n_neuronal} above zero; "
          f"glial {r.median_glial:+.3f}, {r.n_glial_above0}/{r.n_glial}); "
          f"null |T| never exceeded {r.T_null_max:.3f} in {r.n_draw:,} draws; z = {r.z:,.0f}")

# the null distribution of T, for the figure
pd.concat([pd.DataFrame(dict(sample=s, mode=MODE, rep=np.arange(len(t)), T=t))
           for s, t in tnull_keep.items()], ignore_index=True) \
  .to_csv(OUT / "a3d_local_null_group_null.csv", index=False)

                             0            1
sample                      WT           AD
mode               permutation  permutation
n_neuronal                  17           17
n_glial                     39           39
median_neuronal       0.169161     0.092705
median_glial         -0.501839    -0.395116
n_neuronal_above0           14           10
n_glial_above0               3            2
T_obs                    0.671     0.487821
n_draw                  100000       100000
T_null_mean          -0.000027    -0.000014
T_null_sd             0.006608     0.009005
T_null_max            0.037127     0.044598
z                   101.552618    54.171641
p                      0.00001      0.00001

[WT] T = +0.671  (neuronal +0.169, 14/17 above zero; glial -0.502, 3/39); null |T| never exceeded 0.037 in 100,000 draws; z = 102
[AD] T = +0.488  (neuronal +0.093, 10/17 above zero; glial -0.395, 2/39); null |T| never exceeded 0.045 in 100,000 draws; z = 54


## 6. Correctness gates

On by default. They run last, so every table above is already written; a failure here invalidates
the tables rather than preventing them from existing.

**(c) and (e) are the two that matter**, and they check different things.

**(c) asks whether the closed form is the right formula.** Sections 3–5 compute everything from
$E_g$ and $V_g$; if those expressions were wrong, every number in this notebook would be wrong in a
way no amount of internal consistency would reveal. So gate (c) does not check the algebra against
more algebra. It takes a few thousand *real* bins, physically pools each one's granule and residual
transcripts, shuffles which of them are "granule", and counts genes — a thousand times. If the
closed form describes that shuffle, the simulated per-gene mean lands on $E_g$ within Monte-Carlo
error and the simulated spread lands on $\sqrt{V_g}$. It also confirms that every shuffle preserves
the granule transcript total exactly: the null moves composition, never abundance.

**(e) asks whether the machinery manufactures results.** It splits each bin's *residual*
transcripts at random into two halves and feeds one half in as though it were the granule layer.
Those halves genuinely are random samples of one another, so the null must accept them: the
z-scores have to be centred on zero with unit spread, and essentially no gene may come out
significant. If this reports significance here, it would report significance on anything.

In [7]:
if VALIDATE:
    # ---- (a) the 10 um bins re-aggregate to A3c's transcript-level partition, gene by gene ----
    pc = pd.read_csv(C.A3C_DIR / "partition_counts.csv")
    ref = (pc[pc["gene"].isin(GENES)]
           .melt(id_vars=["gene", "sample"], value_vars=["granule", "residual_extrasomatic"],
                 var_name="layer", value_name="n_ref"))
    got = (binned.groupby(["sample", "layer", "gene"], observed=True)["n"].sum()
           .rename("n_got").reset_index())
    for col in ("sample", "layer", "gene"):
        got[col] = got[col].astype(str)      # categorical vs object merges to NaN, silently
    chk = ref.merge(got, on=["sample", "layer", "gene"], how="outer").fillna(0)
    bad = chk[chk["n_ref"] != chk["n_got"]]
    assert bad.empty, f"(a) bin counts disagree with partition_counts.csv:\n{bad.head(10)}"
    print(f"[ok] (a) {len(chk):,} gene x layer x sample totals match partition_counts.csv exactly")

    # ---- (b) nothing that defined or filtered a granule is in the gene set ----
    tainted = set(GENES) & (set(C.SYN_GENES) | set(A3.load_nc_genes()))
    assert not tainted, f"(b) seed or NC genes reached the null: {sorted(tainted)}"
    assert len(GENES) == 252, f"(b) expected 252 neutral genes, got {len(GENES)}"
    print(f"[ok] (b) {len(GENES)} genes, none of the {len(C.SYN_GENES)} seeds or "
          f"{len(A3.load_nc_genes())} negative controls among them")

    # ---- (c) BRUTE FORCE: shuffle labels in real bins, against the closed form ----
    # The check the whole notebook rests on. Not algebra against algebra -- real transcripts,
    # physically relabelled. See a3_common.local_null_permutation_check.
    cal = []
    for s in C.SAMPLES:
        rng = np.random.default_rng(zlib.crc32(f"{C.LOCAL_NULL_SEED}|bruteforce|{s}".encode()))
        d = A3.local_null_permutation_check(mats[s]["O"], mats[s]["R"], rng=rng)
        d.insert(0, "sample", s)
        d["gene"] = np.asarray(GENES)[d["gene_index"].to_numpy()]
        cal.append(d)
    cal = pd.concat(cal, ignore_index=True)
    cal.to_csv(OUT / "a3d_local_null_calibration.csv", index=False)

    zmax = cal["z_of_mean_diff"].abs().max()
    rlo, rhi = cal["sd_ratio"].min(), cal["sd_ratio"].max()
    assert bool(cal["granule_total_ok"].all()), "(c) a shuffle changed the granule transcript total"
    assert zmax < 6.0, f"(c) shuffled mean differs from the closed form, max |z| = {zmax:.2f}"
    assert 0.85 < rlo and rhi < 1.15, f"(c) shuffled sd / closed-form sd = {rlo:.3f} .. {rhi:.3f}"
    print(f"[ok] (c) physically shuffling labels in "
          f"{int(cal['n_bin_check'].iloc[0]):,} real bins x {int(cal['n_rep'].iloc[0]):,} reps "
          f"reproduces the closed form: max |z| of the mean {zmax:.2f} "
          f"(expect <4 over {len(cal):,} genes), sd ratio {rlo:.3f}..{rhi:.3f}")
    print("[ok] (c) every shuffle preserved the granule transcript total exactly "
          "-- the null moves composition, never abundance")

    # ---- (d) NEGATIVE CONTROL: split the residual pool and test one half against the other ----
    neg = []
    for s in C.SAMPLES:
        R = mats[s]["R"]
        rng = np.random.default_rng(zlib.crc32(f"{C.LOCAL_NULL_SEED}|split|{s}".encode()))
        half = rng.binomial(R.data.astype(np.int64), 0.5).astype(float)
        A_ = sparse.csr_matrix((half, R.indices, R.indptr), shape=R.shape)
        Bm = sparse.csr_matrix((R.data - half, R.indices, R.indptr), shape=R.shape)
        k = (np.asarray(A_.sum(axis=1)).ravel() > 0) & \
            (np.asarray(Bm.sum(axis=1)).ravel() >= MIN_POOL)
        A_, Bm = A_[k], Bm[k]
        obs = np.asarray(A_.sum(axis=0)).ravel()
        E, V = A3.local_null_moments(A_, Bm, MODE)
        sd = np.sqrt(V)
        z = np.divide(obs - E, sd, out=np.zeros_like(E), where=sd > 0)
        neg.append(dict(sample=s, mode=MODE, n_bin=int(k.sum()),
                        mean_z=float(z.mean()), sd_z=float(z.std(ddof=1)),
                        max_abs_z=float(np.abs(z).max()),
                        frac_fdr05=float((A3.bh_fdr(2 * norm.sf(np.abs(z))) < 0.05).mean())))
    neg = pd.DataFrame(neg)
    neg.to_csv(OUT / "a3d_local_null_negative_control.csv", index=False)
    print("\n[d] two random halves of the residual pool, tested against each other:")
    print(neg.to_string(index=False))
    for r in neg.itertuples():
        assert abs(r.mean_z) < 0.25, f"(d) [{r.sample}] z centred at {r.mean_z:+.3f}, not 0"
        assert 0.80 < r.sd_z < 1.25, f"(d) [{r.sample}] z spread {r.sd_z:.3f}, not 1"
        assert r.frac_fdr05 < 0.01, (f"(d) [{r.sample}] {r.frac_fdr05:.1%} of genes came out "
                                    f"significant on exchangeable material")
    print("\n[ok] (d) on material that genuinely IS a random sample of itself, the null accepts: "
          "z centred on zero with unit spread and no gene called. The significance in section 4 "
          "is therefore a property of granules, not of the machinery.")

    # ---- (e) the seed reproduces ----
    for s in C.SAMPLES:
        again = A3.local_null_permutation_check(
            mats[s]["O"], mats[s]["R"], n_rep=5,
            rng=np.random.default_rng(zlib.crc32(f"{C.LOCAL_NULL_SEED}|repro|{s}".encode())),
            verbose=False)
        twice = A3.local_null_permutation_check(
            mats[s]["O"], mats[s]["R"], n_rep=5,
            rng=np.random.default_rng(zlib.crc32(f"{C.LOCAL_NULL_SEED}|repro|{s}".encode())),
            verbose=False)
        assert again["mc_mean"].equals(twice["mc_mean"]), f"(e) [{s}] the null is not reproducible"
    print("[ok] (e) crc32-seeded shuffles reproduce across kernel restarts")

[ok] (a) 1,008 gene x layer x sample totals match partition_counts.csv exactly
[ok] (b) 252 genes, none of the 20 seeds or 19 negative controls among them
    brute force: 1,000 shuffles over 2,000 real bins (499,312 transcripts)


    brute force: 1,000 shuffles over 2,000 real bins (359,748 transcripts)


[ok] (c) physically shuffling labels in 2,000 real bins x 1,000 reps reproduces the closed form: max |z| of the mean 3.24 (expect <4 over 504 genes), sd ratio 0.938..1.097
[ok] (c) every shuffle preserved the granule transcript total exactly -- the null moves composition, never abundance



[d] two random halves of the residual pool, tested against each other:
sample        mode  n_bin    mean_z     sd_z  max_abs_z  frac_fdr05
    WT permutation 143813 -0.021566 1.040500   2.897609         0.0
    AD permutation 112874  0.028552 0.885842   3.139376         0.0

[ok] (d) on material that genuinely IS a random sample of itself, the null accepts: z centred on zero with unit spread and no gene called. The significance in section 4 is therefore a property of granules, not of the machinery.


[ok] (e) crc32-seeded shuffles reproduce across kernel restarts


## Outputs

| file | contents |
| --- | --- |
| `a3d_bin_layer_counts.parquet` | the 10 µm grid: transcript counts per bin × gene × compartment, both sections, 252 neutral genes |
| `a3d_local_null_scope.csv` | grid and null settings, bins occupied / granule-bearing / kept, and the fraction of the granule layer the kept bins cover |
| `a3d_local_null_genes.csv` | per gene × section: observed and expected granule counts, log2 ratio, z, p, FDR, and the panel's cell-type, synapse and neuropil labels |
| `a3d_local_null_group.csv` | the neuronal-minus-glial contrast T, observed against its null, per section |
| `a3d_local_null_group_null.csv` | the null distribution of T, one row per parametric draw — for the figure |
| `a3d_local_null_calibration.csv` | gate (c): per gene, the brute-force shuffled mean and sd against the closed-form ones |
| `a3d_local_null_negative_control.csv` | gate (d): z-score centre and spread when two random halves of the residual pool are tested against each other |

Figures are drawn by `Rscript A3_figures.R`, which reads these and writes
`output/figures/a3d_local_null.jpeg`.

The pseudo-granule counterpart to this analysis — the same hypothesis built physically and fed back
through the detector — is `A3e_pseudo_granules.ipynb`.